### Fuentes de datos utilizadas
Incluyendo la descarga de soportes adicionales (diccionarios de metadatos y relación variable significado).

| Fuente | Descripción | URL |
|---|---|---|
|Conducta suicida Bogotá (OSB) | Casos de ideación, intento y suicidio consumado 2023-2025| https://datosabiertos.bogota.gov.co/dataset/tasa-de-suicidio-en-bogota-d-c |
| Encuesta Distrital de Percepción 2025 | Determinantes sociales de salud percibida | https://www.sdp.gov.co/gestion-estudios-estrategicos/informacion-estadisticas/encuesta-distrital-percepcion |
| Sistema Distrital de Parques y Escenarios Públicos<br>Indicador Espacio Público Ciudad. Bogotá D.C. | Inventario de parques y espacio público por localidad | https://datosabiertos.bogota.gov.co/dataset/sistema-distrital-de-parques-y-escenarios-publicos-deportivos<br>https://datosabiertos.bogota.gov.co/dataset/indicador-espacio-publico-ciudad-bogota-d-c  |
| Limites Político-Administrativos (UPL y Localidades) | Límites geográficos oficiales de las localidades de Bogotá | https://datosabiertos.bogota.gov.co/dataset/localidad-bogota-d-c |


## Preprocesamiento de los datos primarios.

```
data_jam_bogota_2026/
	outputs/
		tables/
		figures/
	notebooks/
		001.ipynb
		002.ipynb
	data/
		raw_zips/
			raw.rar # Contiene las carpetas 01, 02, 03, 04. Donde cada uno representan los datasets
		raw/
			01/
			02/
			03/
			04/
		processed/
```



### 1. Instalación de dependencias.

In [1]:
import os
import warnings
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from google.colab import drive
warnings.filterwarnings("ignore")

In [2]:
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# 1.1 Definición de rutas del proyecto
PROJECT_PATH = "/content/drive/MyDrive/DataJAM_Bogota_2026/"
RAW_DATA_PATH = os.path.join(PROJECT_PATH, "data/raw/")

PROCESSED_DATA_PATH = os.path.join(PROJECT_PATH, "data/processed/")
OUTPUTS_FIG_PATH = os.path.join(PROJECT_PATH, "outputs/figures/")
OUTPUTS_TABLE_PATH = os.path.join(PROJECT_PATH, "outputs/tables/")

RAW_ZIP_PATH = f"{PROJECT_PATH}data/raw_zips/raw.rar"
DESTINATION = os.path.join(PROJECT_PATH, "data/raw")

for path in [
    RAW_DATA_PATH,
    PROCESSED_DATA_PATH,
    OUTPUTS_FIG_PATH,
    OUTPUTS_TABLE_PATH,
]:
    os.makedirs(path, exist_ok=True)

print(f"Inicialización del Entorno 001 completada.")

Inicialización del Entorno 001 completada.


### Extracción del conjunto de datos espaciales y tabulares (.csv, .geojson).

In [ ]:
!unrar x $RAW_ZIP_PATH $DESTINATION


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/drive/MyDrive/data_jam_bogota_2026/data/raw_zips/raw.rar

Creating    /content/drive/MyDrive/data_jam_bogota_2026/data/raw/01   OK
Extracting  /content/drive/MyDrive/data_jam_bogota_2026/data/raw/01/conducta_suicida_bogota.csv      24%  OK 
Extracting  /content/drive/MyDrive/data_jam_bogota_2026/data/raw/01/metadato-osb_salud-mental_suicidio_consumado.csv      24%  OK 
Extracting  /content/drive/MyDrive/data_jam_bogota_2026/data/raw/01/metadato_osb_salud_mental_ideacion_e_intento.csv      24%  OK 
Extracting  /content/drive/MyDrive/data_jam_bogota_2026/data/raw/01/suicidio_consumado_bogota.csv      24%  OK 
Creating    /content/drive/MyDrive/data_jam_bogota_2026/data/raw/02   OK
Extracting  /content/drive/MyDrive/data_jam_bogota_2026/data/raw/02/diccionario_base.xlsx      25%  OK 
Extracting  /content/drive/MyDrive/data_jam_bogota_2026/data/r

Creación de función auxiliar para detectar de forma automática la codificación y el delimitador de cada archivo de extensión ".csv".

In [4]:
def load_district_csv(filepath: str) -> pd.DataFrame:
    """
    Carga archivos CSV comprobando las codificaciones y los delimitadores estándar de los datos públicos colombianos
    """
    encodings = ["latin1", "utf-8-sig", "utf-8", "cp1252"]
    delimiters = [";", ","]

    for enc in encodings:
        for sep in delimiters:
            try:
                df = pd.read_csv(
                    filepath,
                    sep=sep,
                    encoding=enc,
                    low_memory=False,
                    on_bad_lines="skip",
                )
                if df.shape[1] > 1:
                    return df
            except (UnicodeDecodeError, pd.errors.ParserError):
                continue

    return pd.read_csv(
        filepath,
        sep=None,
        engine="python",
        encoding="latin1",
        on_bad_lines="skip",
    )

### 2. Carga y exploración de fuentes primarias.
Cada uno de los archivos fueron renombrados para establecer una estructura semejante a la tabla X, lo que permite una mejor interpretación del código.

In [5]:
# 2.1 Indicador y Eventos de Conducta Suicida (SaluData / SDS)
suicidal_behavior_df = load_district_csv(
    os.path.join(RAW_DATA_PATH, "01/conducta_suicida_bogota.csv")
)

completed_suicide_df = load_district_csv(
    os.path.join(RAW_DATA_PATH, "01/suicidio_consumado_bogota.csv")
)

# 2.2 Encuesta Distrital de Percepción 2025 (SDP)
perception_survey_df = load_district_csv(
    os.path.join(RAW_DATA_PATH, "02/encuesta_percepcion_2025.csv")
)

# 2.3 Capas de Zonas Verdes y Espacio Público
# 2.3.1 Indicador Espacio Público Ciudad (DADEP)
public_space_indicator_gdf = gpd.read_file(
    os.path.join(RAW_DATA_PATH, "03/indicador_espacio_publico_dadep.geojson")
).to_crs(epsg=4326)

# 2.3.2 Sistema Distrital de Parques y Escenarios Públicos Deportivos (IDRD)
parks_system_gdf = gpd.read_file(
    os.path.join(RAW_DATA_PATH, "03/sistema_parques_bogota.geojson")
).to_crs(epsg=4326)

# 2.4 Límites Político-Administrativos (Localidades / IDECA - SDP)
localities_gdf = gpd.read_file(
    os.path.join(RAW_DATA_PATH, "04/localidades_bogota.geojson")
).to_crs(epsg=4326)

print(
    "Conjuntos de datos primarios correctamente cargados para etapa de preprocesamiento."
)

Conjuntos de datos primarios correctamente cargados para etapa de preprocesamiento.


Auxiliares

In [6]:
# 1. Función maestra para corregir problemas de codificación (Mojibake)
def fix_mojibake_encoding(text: str) -> str:
    if pd.isna(text):
        return text
    # Se incluyen los caracteres rotos de Ciudad Bolívar y San Cristóbal que identificaste
    replacements = {
        "Ã±": "ñ", "Ã‘": "Ñ", "Ã¡": "á", "Ã©": "é", "Ã": "í",
        "Ã³": "ó", "Ãº": "ú", "Ã": "Á", "Ã‰": "É", "Ã": "Í",
        "Ã“": "Ó", "Ãš": "Ú", "¥": "Ñ", "Í\xad": "í", "Í³": "ó"
    }
    text_str = str(text)
    for bad_char, good_char in replacements.items():
        text_str = text_str.replace(bad_char, good_char)
    return text_str.strip()

# 2. Función para estandarizar identificadores (mayúsculas y sin tildes)
def clean_text_identifier(text: str) -> str:
    if pd.isna(text):
        return np.nan
    # Primero reparamos la codificación, luego estandarizamos
    text = fix_mojibake_encoding(text)
    text = str(text).upper().strip()
    accent_map = {
        "Á": "A", "É": "E", "Í": "I", "Ó": "O", "Ú": "U", "Ü": "U", "Ñ": "N",
    }
    for accented, standard in accent_map.items():
        text = text.replace(accented, standard)
    if text == "CANDELARIA":
        text = "LA CANDELARIA"
    return text

In [7]:
# 1. Limpieza estructural de los DataFrames tabulares
for df in [completed_suicide_df, perception_survey_df, suicidal_behavior_df]:
    # Normalizar encabezados (quitar BOM ï»¿, espacios extra y pasar a mayúsculas)
    df.columns = df.columns.str.replace("ï»¿", "").str.strip().str.upper()

    # Corregir textos en todas las columnas categóricas desde el origen
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].apply(fix_mojibake_encoding)

print("Encabezados normalizados y codificación de caracteres reparada en los DataFrames primarios.")

Encabezados normalizados y codificación de caracteres reparada en los DataFrames primarios.


###2.1 Normalización de Claves Geográficas.

In [8]:
loc_name_col = [
    c
    for c in localities_gdf.columns
    if "NOM" in c.upper() or "LOC" in c.upper()
][0]
loc_code_col = [c for c in localities_gdf.columns if "COD" in c.upper()][0]

localities_gdf["locality_clean"] = localities_gdf[loc_name_col].apply(
    clean_text_identifier
)
localities_gdf["locality_code"] = (
    localities_gdf[loc_code_col].astype(str).str.extract(r"(\d+)")[0].str.zfill(2)
)

print(f"Claves geográficas (nombre y código de 2 dígitos) estandarizadas en localidades.")

Claves geográficas (nombre y código de 2 dígitos) estandarizadas en localidades.


### 3. Cálculo de indicadores epidemiológicos.

In [9]:
suicide_df = completed_suicide_df.copy()

# 3.1 Identificación de columnas de localidad y sexo en los microdatos
loc_col = [
    c
    for c in suicide_df.columns
    if "LOC" in c.upper() or "MUNICIPIO" in c.upper()
][0]
sex_col = [c for c in suicide_df.columns if "SEX" in c.upper()][0]

# 3.2 Extracción del código numérico de 2 dígitos de la localidad para posterior estandarización
suicide_df["locality_code"] = (
    suicide_df[loc_col].astype(str).str.extract(r"(\d+)")[0].str.zfill(2)
)

# 3.3 Mapeo del código numérico al nombre oficial de la localidad usando localities_gdf
loc_mapping = dict(
    zip(localities_gdf["locality_code"], localities_gdf["locality_clean"])
)
suicide_df["locality_clean"] = suicide_df["locality_code"].map(loc_mapping)

# Aplicación de filtro unicamente a las 20 localidades oficiales validas de Bogotá
suicide_clean_df = suicide_df[suicide_df["locality_clean"].notna()].copy()

# 3.4 Conteo total de eventos por localidad
suicide_summary_df = (
    suicide_clean_df.groupby("locality_clean")
    .size()
    .reset_index(name="historico_total_suicidios_localidad")
)

# 3.5 Tabulación cruzada por Sexo
sex_pivot_df = (
    pd.crosstab(
        suicide_clean_df["locality_clean"], suicide_clean_df[sex_col]
    )
    .reset_index()
    .rename_axis(None, axis=1)
    .rename(columns={
        "Hombre": "hombres",
        "Mujer": "mujeres"
    }) # Estandarización a minúsculas, al igual que el resto de nombre de columnas
)

# 3.6 Consolidación de la tabla epidemiológica final
epidemiological_summary_df = suicide_summary_df.merge(
    sex_pivot_df, on="locality_clean", how="left"
)

print(f"Indicadores epidemiológicos consolidados exitosamente para {len(epidemiological_summary_df)} localidades oficiales.")
epidemiological_summary_df.head(20)

Indicadores epidemiológicos consolidados exitosamente para 20 localidades oficiales.


,locality_clean,historico_total_suicidios_localidad,hombres,mujeres
0,ANTONIO NARINO,45,28,17
1,BARRIOS UNIDOS,80,63,17
2,BOSA,325,253,72
3,CHAPINERO,158,129,29
4,CIUDAD BOLIVAR,400,314,86
5,ENGATIVA,368,271,97
6,FONTIBON,161,125,36
7,KENNEDY,477,350,127
8,LA CANDELARIA,17,12,5
9,LOS MARTIRES,61,51,10


### 4. Resultados Agregados de la Encuesta de Percepción 2025 (SDP).

In [10]:
# 4.1 Estandarización de los encabezados de las columnas
perception_survey_df.columns = [
    c.upper().strip() for c in perception_survey_df.columns
]

# 4.2 Identificación de la columna "localidad" en los microdatos de la encuesta de percepción
survey_loc_col = [
    c
    for c in perception_survey_df.columns
    if "LOC" in c or "LOCALIDAD" in c or "COD_LOC" in c
][0]
perception_survey_df["locality_clean"] = perception_survey_df[
    survey_loc_col
].apply(clean_text_identifier)

# 4.3 Indicadores agregados de la encuesta por "localidad"
survey_summary_df = (
    perception_survey_df.groupby("locality_clean")
    .agg(total_surveyed_sample=(survey_loc_col, "count"))
    .reset_index()
)

print("Se han agregado los indicadores de la encuesta de percepción por 'localidad'.")
survey_summary_df.head()

Se han agregado los indicadores de la encuesta de percepción por 'localidad'.


,locality_clean,total_surveyed_sample
0,1,834
1,10,1232
2,11,1802
3,12,270
4,13,335


### 5. Análisis Espacial: Espacios Verdes por Localidad.

In [11]:
# 5.1 Función para corregir caracteres dañados específicos encontrados (ej. ¥)
def fix_park_text(text: str) -> str:
    if pd.isna(text):
        return text
    text_str = str(text).replace("¥", "Ñ").replace("Ã±", "Ñ")
    return clean_text_identifier(text_str)


# 5.2 Crear copia del dataset de parques y normalizar nombres
parks_clean_df = parks_system_gdf.copy()
parks_clean_df["locality_clean"] = parks_clean_df["LocNombre"].apply(
    fix_park_text
)
locality_name_fixes = { # Correción de problematica entre nombres de la misma localidad en distintas base de datos
    "MARTIRES": "LOS MARTIRES",
    "RAFAEL URIBE": "RAFAEL URIBE URIBE",
    "SANTAFE": "SANTA FE",
}
parks_clean_df["locality_clean"] = parks_clean_df["locality_clean"].replace(locality_name_fixes)

# 5.3 Convertir área a formato numérico
if "SHAPE_Area" in parks_clean_df.columns:
    parks_clean_df["area_sq_meters"] = pd.to_numeric(
        parks_clean_df["SHAPE_Area"], errors="coerce"
    ).fillna(0)
else:
    parks_projected = parks_clean_df.to_crs(epsg=9377)
    parks_clean_df["area_sq_meters"] = parks_projected.geometry.area

# 5.4 Agrupar métricas por las localidades oficiales
parks_summary_df = (
    parks_clean_df.groupby("locality_clean")
    .agg(
        parks_count=("area_sq_meters", "count"),
        total_green_area_sqm=("area_sq_meters", "sum"),
    )
    .reset_index()
)

print(f"Métricas espaciales consolidadas para {len(parks_summary_df)} localidades de Bogotá.")
parks_summary_df.head(20)

Métricas espaciales consolidadas para 20 localidades de Bogotá.


,locality_clean,parks_count,total_green_area_sqm
0,ANTONIO NARINO,55,2.901663e+05
1,BARRIOS UNIDOS,123,1.742276e+06
2,BOSA,251,1.222724e+06
3,CHAPINERO,160,7.952233e+05
4,CIUDAD BOLIVAR,447,2.212691e+06
5,ENGATIVA,552,6.179346e+06
6,FONTIBON,280,1.609459e+06
7,KENNEDY,552,3.792798e+06
8,LA CANDELARIA,10,1.946796e+04
9,LOS MARTIRES,47,2.032605e+05


### 6. Integración y Exportación de los Datasets Principales.

In [12]:
# 6.2 Normalizar encabezados (quitar BOM ï»¿, espacios extra y pasar a mayúsculas)
completed_suicide_df.columns = (
    completed_suicide_df.columns.str.replace("ï»¿", "")
    .str.strip()
    .str.upper()
)
perception_survey_df.columns = (
    perception_survey_df.columns.str.replace("ï»¿", "")
    .str.strip()
    .str.upper()
)

In [13]:
# 6.3 Corregir textos en columnas tipo objeto
for col in completed_suicide_df.select_dtypes(include="object").columns:
    completed_suicide_df[col] = completed_suicide_df[col].apply(
        fix_mojibake_encoding
    )

for col in perception_survey_df.select_dtypes(include="object").columns:
    perception_survey_df[col] = perception_survey_df[col].apply(
        fix_mojibake_encoding
    )

In [14]:
# 6.4 Fusión de capas territoriales en el mapa maestro
master_gdf = localities_gdf[["locality_clean", "geometry"]].merge(
    parks_summary_df, on="locality_clean", how="left"
)
master_gdf = master_gdf.merge(
    survey_summary_df, on="locality_clean", how="left"
)

master_gdf["parks_count"] = master_gdf["parks_count"].fillna(0)
master_gdf["total_green_area_sqm"] = master_gdf["total_green_area_sqm"].fillna(
    0
)

In [15]:
# 6.5 Exportación de todos los archivos procesados a data/processed/
master_gdf.to_file(
    os.path.join(
        PROCESSED_DATA_PATH, "master_mental_health_bogota_2025.geojson"
    ),
    driver="GeoJSON",
)
master_gdf.drop(columns="geometry").to_csv(
    os.path.join(PROCESSED_DATA_PATH, "master_mental_health_bogota_2025.csv"),
    index=False,
)

# --- 1. PREPROCESAMIENTO FINAL: EVENTOS DE SUICIDIO ---
# Identificamos la columna de localidad y extraemos su código numérico
loc_col_final = [c for c in completed_suicide_df.columns if "LOC" in c.upper() or "MUNICIPIO" in c.upper()][0]
completed_suicide_df["temp_code"] = completed_suicide_df[loc_col_final].astype(str).str.extract(r"(\d+)")[0].str.zfill(2)

# [CORRECCIÓN]: Reemplazar los textos originales por los nombres estandarizados
# (Mayúsculas y sin tildes) cruzando con el código de la localidad
loc_mapping = dict(zip(localities_gdf["locality_code"], localities_gdf["locality_clean"]))
completed_suicide_df[loc_col_final] = completed_suicide_df["temp_code"].map(loc_mapping)

# Extracción de la variable temporal (Año del hecho)
if "ANO_DEL_HECHO" in completed_suicide_df.columns:
    completed_suicide_df["year"] = pd.to_numeric(completed_suicide_df["ANO_DEL_HECHO"], errors="coerce")

# Filtramos solo los códigos correspondientes a las 20 localidades oficiales (01 al 20)
valid_localities = [str(i).zfill(2) for i in range(1, 21)]
suicide_final_export = completed_suicide_df[completed_suicide_df["temp_code"].isin(valid_localities)].drop(columns=["temp_code"])

# Exportamos el dataframe ya filtrado y sin caracteres rotos
suicide_final_export.to_csv(
    os.path.join(PROCESSED_DATA_PATH, "suicide_events_clean.csv"),
    index=False,
    encoding="utf-8-sig",
)

# --- 2. PREPROCESAMIENTO FINAL: ENCUESTA DE PERCEPCIÓN ---
# Casteo de tipos numéricos y homologación de códigos de error (98/99 -> NaN)
model_vars = ["A3", "A4", "A5", "A6X2"]
for col in model_vars:
    if col in perception_survey_df.columns:
        perception_survey_df[col] = pd.to_numeric(perception_survey_df[col], errors="coerce")
        # Convertimos a nulo (NaN) los valores >= 90 (No sabe/No responde)
        perception_survey_df.loc[perception_survey_df[col] >= 90, col] = np.nan

perception_survey_df.to_csv(
    os.path.join(PROCESSED_DATA_PATH, "perception_survey_clean.csv"),
    index=False,
    encoding="utf-8-sig",
)

print(f"Todos los conjuntos de datos limpios han sido exportados a: {PROCESSED_DATA_PATH}")

Todos los conjuntos de datos limpios han sido exportados a: /content/drive/MyDrive/DataJAM_Bogota_2026/data/processed/
